In [0]:
bronze_path   = 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/projet1/bronze/'
silver_path   = 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/projet1/silver/'
gold_path     = 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/projet1/gold/'
resource_path = '/Volumes/bikestore/resource/01-origem/'
resource_path_volume = "bikestore.bronze"


In [0]:
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_brands=spark.read.csv(f'{resource_path}/brands.csv',
                          header=True, 
                          inferSchema=True
                          )
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_categories=spark.read.csv(f'{resource_path}/categories.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_customers=spark.read.csv(f'{resource_path}/customers.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_order_items=spark.read.csv(f'{resource_path}/order_items.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_orders=spark.read.csv(f'{resource_path}/orders.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_products=spark.read.csv(f'{resource_path}/products.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_staffs=spark.read.csv(f'{resource_path}/staffs.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_stocks=spark.read.csv(f'{resource_path}/stocks.csv',
                          header=True, 
                          inferSchema=True
                          )
						  
# Ler o CSV ou origem qualquer em DataFrame do Spark
df_stores=spark.read.csv(f'{resource_path}/stores.csv',
                          header=True, 
                          inferSchema=True
                          )
						  


In [0]:
# Write DataFrames to Unity Catalog managed tables
(
    df_brands.write
        .mode('overwrite')
    .format('delta')
    .option("mergeSchema", "true") 
    .saveAsTable(f"{resource_path_volume}.brands")        
)
    
df_categories.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.categories")

df_customers.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.customers")

df_order_items.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.order_items")

df_orders.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.orders")

df_products.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.products")

df_staffs.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.staffs")

df_stocks.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.stocks")

df_stores.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{resource_path_volume}.stores")    



In [0]:
# Secrets cannot be created from notebooks - use Databricks CLI instead:
# databricks secrets create-scope --scope bnptec
# databricks secrets put-secret --scope bnptec --key sendgrid-key --string-value "SUA_API_KEY"

# To retrieve a secret in your notebook, use:
# api_key = dbutils.secrets.get(scope="bnptec", key="sendgrid-key")

print("To create secrets, run the CLI commands above from your terminal.")

In [0]:
import requests

def enviar_email(status, mensagem):

    api_key = dbutils.secrets.get(scope="bnptec", key="sendgrid-key")

    subject = f"🚀 Databricks Pipeline - {status}"

    html = f"""
    <html>
        <body style="font-family: Arial;">
            <h2>Status: {status}</h2>
            <p>{mensagem}</p>
            <hr>
            <p><b>Ambiente:</b> Databricks</p>
        </body>
    </html>
    """

    body = {
        "personalizations": [
            {
                "to": [{"email": "jose.paulo.jorge.santos@gmail.com"}],
                "subject": subject
            }
        ],
        "from": {"email": "seu_email@dominio.com"},
        "content": [
            {
                "type": "text/html",
                "value": html
            }
        ]
    }

    response = requests.post(
        "https://api.sendgrid.com/v3/mail/send",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json=body
    )

    print("Email status:", response.status_code)

In [0]:
try:
    # ============================
    # SEU PIPELINE AQUI
    # ============================

    (
        df_brands.write
            .mode('overwrite')
        .format('delta')
        .option("mergeSchema", "true") 
        .saveAsTable(f"{resource_path_volume}.brands")        
    )

    # ============================
    # Verificar se o secret existe antes de enviar email
    try:
        enviar_email("SUCESSO", "Pipeline executado com sucesso!")
    except Exception as email_error:
        print(f"Aviso: Não foi possível enviar email de notificação: {email_error}")
        print("Pipeline executado com sucesso, mas email não configurado.")

except Exception as e:
    erro = str(e)
    
    # Tentar enviar email de erro, mas não falhar se não conseguir
    try:
        enviar_email("ERRO", erro)
    except:
        print(f"Aviso: Não foi possível enviar email de erro. Erro original: {erro}")

    raise e  # importante para o job falhar

In [0]:
dbutils.secrets.put(scope="bnptec", key="sendgrid-key", string_value="SUA_API_KEY")